# CodeSearch-FT Phase 4: Google Colab GPU pilot

This notebook delegates all data preparation, leakage handling, training, validation, checkpointing, and export to the existing `codesearch_ft` package. It uses the pinned CodeSearchNet Python train/validation/test partitions and pinned CodeBERT revision. Nothing trains until you run the dedicated smoke or pilot cell.

**Start here:** Runtime → Change runtime type → T4 GPU. Run the cells in order. Drive mounting is an explicit, user-run cell. Keep the bundle ZIP on your computer.

## 1. Verify GPU and CUDA availability
Run this first after selecting a GPU runtime.

In [ ]:
import sys

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Select a GPU runtime in Colab, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Mount Google Drive for persistent outputs
Run this cell yourself and approve Colab's Drive access prompt. Checkpoints are written to Drive so they survive runtime loss.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
PERSIST_ROOT = Path("/content/drive/MyDrive/codesearch_ft_colab_pilot")
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent output root:", PERSIST_ROOT)

## 3. Upload and extract the deployment ZIP
Choose `artifacts/colab/codesearch_ft_colab_bundle.zip` from your computer.

In [ ]:
import shutil
import zipfile

from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one Colab deployment ZIP.")
BUNDLE_ZIP = Path("/content") / zip_names[0]
BUNDLE_ROOT = Path("/content/codesearch_ft_colab_bundle")
if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)
BUNDLE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    zf.extractall(BUNDLE_ROOT)
print("Bundle extracted to", BUNDLE_ROOT)

## 4. Verify integrity and Python compatibility
Only the wheel, YAML, dependency requirements, and manifest should be in the ZIP.

In [ ]:
import hashlib
import json
import zipfile
from email.parser import Parser

from packaging.specifiers import SpecifierSet
from packaging.version import Version

manifest = json.loads((BUNDLE_ROOT / "integrity-manifest.json").read_text())
expected = {r["path"] for r in manifest["files"]} | {"integrity-manifest.json"}
with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    actual = set(zf.namelist())
if actual != expected:
    raise ValueError(f"Unexpected ZIP contents: {actual ^ expected}")


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


for record in manifest["files"]:
    path = BUNDLE_ROOT / record["path"]
    if sha256_file(path) != record["sha256"]:
        raise ValueError(f"Hash mismatch: {path.name}")
if not (sys.version_info >= (3, 11) and sys.version_info < (3, 14)):
    raise RuntimeError(f"Wheel requires Python {manifest['python_requires']}; got {sys.version}")
with zipfile.ZipFile(next(BUNDLE_ROOT.glob("codesearch_ft-*.whl"))) as wheel:
    metadata_name = next(n for n in wheel.namelist() if n.endswith(".dist-info/METADATA"))
    requires_python = Parser().parsestr(wheel.read(metadata_name).decode()).get("Requires-Python")
if not requires_python or Version(sys.version.split()[0]) not in SpecifierSet(requires_python):
    raise RuntimeError(f"Wheel Python requirement {requires_python!r} rejects this runtime.")
print("Integrity verified:", sorted(actual))
print("Python compatibility verified:", sys.version.split()[0])

## 5. Install missing training dependencies safely
This resolves compatible packages for the runtime's actual Python. It constrains PyTorch to the exact version already present and does not list Torch in the requirements file.

In [ ]:
import importlib.util
import subprocess
from importlib.metadata import version

wheel_path = next(BUNDLE_ROOT.glob("codesearch_ft-*.whl"))
torch_version = version("torch")
torch_constraints = Path("/content/colab-torch-constraints.txt")
torch_constraints.write_text(f"torch=={torch_version}\n")
req_path = BUNDLE_ROOT / "requirements-colab.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", str(wheel_path)])
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade-strategy",
        "only-if-needed",
        "-c",
        str(torch_constraints),
        "-r",
        str(req_path),
    ]
)
print(
    "Installed project wheel and resolved training dependencies; constrained torch to",
    torch_version,
)

## 6. Training environment and dependency preflight
This validates CUDA, package imports, dependency ranges, and the real CLI config. It does not download CodeBERT or dataset rows and does not train.

In [ ]:
import importlib
import importlib.metadata as md
import json

import torch
from packaging.version import Version

from codesearch_ft.config.loader import load_training_config

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is no longer available.")
if not (
    Version(md.version("torch")) >= Version("2.6") and Version(md.version("torch")) < Version("3")
):
    raise RuntimeError("CodeSearch-FT trainer requires PyTorch >=2.6,<3.")
for module in ("accelerate", "datasets", "mlflow", "transformers", "codesearch_ft"):
    importlib.import_module(module)
if Path(torch.__file__).resolve().is_relative_to(Path("/content").resolve()):
    raise RuntimeError("Torch was shadowed by an uploaded/local package.")
CONFIG_PATH = BUNDLE_ROOT / "train_gpu_pilot.yaml"
base_config = load_training_config(CONFIG_PATH)
subprocess.run(
    ["codesearch", "config", "validate", str(CONFIG_PATH), "--kind", "training"], check=True
)
if (
    base_config.profile != "gpu-pilot"
    or not base_config.max_train_samples
    or base_config.max_train_samples > 10000
):
    raise ValueError("Pilot config must cap training at 10,000 real pairs.")
print(
    json.dumps(
        {
            "python": sys.version,
            "torch": md.version("torch"),
            "torch_cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0),
            "accelerate": md.version("accelerate"),
            "datasets": md.version("datasets"),
            "transformers": md.version("transformers"),
            "mlflow": md.version("mlflow"),
            "wheel_sha256": next(
                r["sha256"] for r in manifest["files"] if r["path"].endswith(".whl")
            ),
            "config_sha256": manifest["config_sha256"],
        },
        indent=2,
    )
)
print("Configuration valid; no training started.")

## 7. Real CodeSearchNet GPU smoke test (explicit run)
This downloads a tiny bounded sample from the pinned real CodeSearchNet Python official train/validation/test splits and runs the package's actual Accelerate trainer. Exact cross-split duplicates are quarantined by the existing adapter. This cell is separate from the bounded pilot.

In [ ]:
import contextlib

import yaml

from codesearch_ft.ml.trainer import train

smoke_dir = PERSIST_ROOT / "smoke"
smoke_dir.mkdir(parents=True, exist_ok=True)
smoke_config = base_config.model_copy(
    update={
        "max_train_samples": 16,
        "validation_max_samples": 8,
        "test_max_samples": 8,
        "micro_batch_size": 2,
        "gradient_accumulation_steps": 1,
        "epochs": 1,
        "output_dir": smoke_dir,
        "tracking_uri": f"sqlite:///{smoke_dir / 'mlflow.db'}",
        "dataset_cache_dir": Path("/content/codesearch_ft_dataset_cache"),
    }
)
smoke_log = smoke_dir / "training.log"
with (
    smoke_log.open("a", encoding="utf-8") as log,
    contextlib.redirect_stdout(log),
    contextlib.redirect_stderr(log),
):
    smoke_result = train(smoke_config)
(smoke_dir / "metrics.json").write_text(json.dumps(smoke_result, indent=2) + "\n")
print(json.dumps(smoke_result, indent=2))
print("Smoke outputs:", smoke_dir)

## 8. Bounded real GPU fine-tuning pilot (run separately)
This is the only full pilot launch cell. It uses no more than 2,000 training pairs, one epoch, micro-batch 4, gradient accumulation 4, and the exact pinned source revisions. The package writes the sampled-data manifest, resume state, exported Transformers model, optimizer/scheduler/RNG state, and metrics directly under Google Drive.

In [ ]:
pilot_dir = PERSIST_ROOT / "pilot"
pilot_dir.mkdir(parents=True, exist_ok=True)
pilot_config = base_config.model_copy(
    update={
        "max_train_samples": 2000,
        "validation_max_samples": 128,
        "test_max_samples": 128,
        "micro_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "epochs": 1,
        "output_dir": pilot_dir,
        "tracking_uri": f"sqlite:///{pilot_dir / 'mlflow.db'}",
        "dataset_cache_dir": Path("/content/codesearch_ft_dataset_cache"),
    }
)
effective = pilot_config.model_dump(mode="json")
effective_yaml = yaml.safe_dump(effective, sort_keys=True)
(pilot_dir / "effective-config.yaml").write_text(effective_yaml, encoding="utf-8")
effective_sha = hashlib.sha256(effective_yaml.encode()).hexdigest()
(pilot_dir / "deployment-metadata.json").write_text(
    json.dumps(
        {
            "python": sys.version,
            "torch": md.version("torch"),
            "torch_cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0),
            "model_id": pilot_config.base_model,
            "model_revision": pilot_config.model_revision,
            "dataset_id": pilot_config.dataset_id,
            "dataset_revision": pilot_config.dataset_revision,
            "config_sha256": effective_sha,
            "bundle_manifest": manifest,
        },
        indent=2,
    )
    + "\n"
)
pilot_log = pilot_dir / "training.log"
with (
    pilot_log.open("a", encoding="utf-8") as log,
    contextlib.redirect_stdout(log),
    contextlib.redirect_stderr(log),
):
    pilot_result = train(pilot_config)
(pilot_dir / "metrics.json").write_text(json.dumps(pilot_result, indent=2) + "\n")
print(json.dumps(pilot_result, indent=2))
print("Persistent pilot artifacts:", pilot_dir)

## 9. Inspect metrics, checkpoints, and logs
Use after smoke or pilot. The export directory contains `model/` with Transformers weights/tokenizer and `metadata.json`; epoch checkpoints include Accelerate optimizer, scheduler, RNG and resume state.

In [ ]:
inspect_dir = PERSIST_ROOT / "pilot"
for rel in ("metrics.json", "dataset-manifest.json", "deployment-metadata.json", "training.log"):
    path = inspect_dir / rel
    print(f"\n--- {path} ---")
    if path.exists():
        print(path.read_text(encoding="utf-8")[-12000:])
print("Checkpoint files:")
for path in sorted(inspect_dir.rglob("*")):
    if path.is_file() and ("checkpoints" in path.parts or "export" in path.parts):
        print(path.relative_to(inspect_dir), path.stat().st_size, "bytes")

## 10. Recover the exported model and resume state
Drive already contains the files. To resume after a lost runtime, rerun setup and preflight, then invoke the package trainer with the latest `checkpoints/epoch-NNN` path and a config whose output directory points to the same Drive pilot directory. The export can be loaded with `AutoModel.from_pretrained(PERSIST_ROOT / 'pilot/export/model')` and `AutoTokenizer.from_pretrained(...)`. Run the optional cell below to download a ZIP of the export. Keep the Drive copy because it also contains optimizer/resume state.

In [ ]:
# Optional: download the Transformers-compatible exported model and tokenizer.
# from google.colab import files
# model_zip = shutil.make_archive(
#     str(PERSIST_ROOT / 'pilot' / 'fine-tuned-codebert'),
#     'zip', root_dir=PERSIST_ROOT / 'pilot' / 'export'
# )
# files.download(model_zip)
# Resume by passing an epoch checkpoint: train(resume_config, resume=checkpoint_path)
